# Higher-Lower: which strategy wins?

Five strategies play **Higher-Lower** (one 52-card deck, one point per correct guess, ties always
win; see `docs/architecture.md` and `HigherLower` for the exact rules):

| Strategy | Idea |
| --- | --- |
| `random` | Guess higher or lower with equal probability. |
| `trivial` | Bet on the larger side of the table card: lower above 7, higher below 7, coin flip on 7. |
| `counting` | Trivial on A-4 and 10-K; on 5-9 follow the count of low (A-4) vs high (10-K) cards seen. |
| `counting advanced` | Same low (A-4) and high (10-K) groups as `counting`. Trivial on those, count on the 7, and on 5, 6, 8, 9 the "delta" (highs seen minus lows seen) decides when to go against the card's own side. See `CountingAdvanced`. |
| `optimal counting` | Counts every rank and bets on the side with more cards left. The best possible solo strategy: the **ground truth** the others are measured against. See `OptimalCounting`. |

Two experiments, both through the Python API (the same runs are available from the command line,
e.g. `python -m advantage_play players=solo_counting`):

1. **Solo**: each strategy goes through the deck alone. This is the cleanest strategy comparison:
   no opponent can influence which cards it faces.
2. **Multi-player**: all five strategies sit at one table. The winner is whoever has most points
   (ties split the win). Seat order matters, because a player inherits the card that beat the
   previous one, so we play **every distinct circular seating** (with one seat fixed, the other
   four can be arranged in 4! = 24 ways) and average.

Every run uses the same seed, so all strategies play the **same deals**. That makes comparisons
*paired*: a strategy's score minus another's, game by game, removes the luck of the deal and is
far more precise than comparing two separate confidence intervals.

In [ ]:
from itertools import permutations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, PercentFormatter

from advantage_play.analysis import mean_ci, win_shares
from advantage_play.games import HigherLower
from advantage_play.games.higher_lower import Guess
from advantage_play.simulation import simulate
from advantage_play.strategies.higher_lower import (
    CardCounting,
    CountingAdvanced,
    OptimalCounting,
    RandomGuess,
    Trivial,
)

N_SOLO = 50_000  # games per strategy, solo
N_MULTI = 50_000  # games per seating order, 5 players
SEED = 0
CONFIDENCE = 0.95

game = HigherLower()
STRATEGIES = {
    "random": RandomGuess(),
    "trivial": Trivial(),
    "counting": CardCounting(),
    "counting advanced": CountingAdvanced(),
    "optimal counting": OptimalCounting(),
}
NAMES = list(STRATEGIES)

## Plot style

One fixed colour per strategy in every chart: `random` is the neutral grey baseline and `optimal counting` the dark ceiling.

In [ ]:
SURFACE, INK, INK_2, AXIS, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#c3c2b7", "#e1e0d9"
COLORS = {
    "random": "#898781",
    "trivial": "#2a78d6",
    "counting": "#eb6834",
    "counting advanced": "#1baf7a",
    "optimal counting": INK,
}
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "figure.dpi": 110,
        "font.family": "sans-serif",
        "text.color": INK,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": INK_2,
        "axes.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.axisbelow": True,
        "axes.titlelocation": "left",
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "xtick.labelsize": 9.5,
        "ytick.labelsize": 9.5,
        "legend.frameon": False,
    }
)


def strategy_legend(fig):
    handles = [Line2D([], [], color=c, lw=2, marker="o", ms=8, label=n) for n, c in COLORS.items()]
    fig.legend(handles=handles, loc="outside lower center", ncol=len(handles), fontsize=10.5)


def summary(table):
    """Mean and confidence-interval half-width of every column."""
    mean, half_width = mean_ci(table.to_numpy(), CONFIDENCE)
    return pd.DataFrame({"mean": mean, "ci": half_width}, index=table.columns)


def gap_to(table, reference, omit=()):
    """Paired difference (strategy minus reference) per strategy, same deals."""
    others = table.drop(columns=[reference, *omit])
    return summary(others.sub(table[reference], axis=0))


def plus_minus(mean, ci):
    return f"{mean:.2f} ± {ci:.2f}"


def signed(mean, ci):
    return f"{mean:+.2f} ± {ci:.2f}"


def percent(mean, ci):
    return f"{mean:.1%} ± {ci:.1%}"


def dot_plot(ax, table, title, xlabel, label, reference=None):
    """One dot per strategy at its mean, its confidence interval as a whisker, value at the tip."""
    y = np.arange(len(table))[::-1]
    for yi, (name, row) in zip(y, table.iterrows(), strict=True):
        lo, hi = row["mean"] - row["ci"], row["mean"] + row["ci"]
        ax.plot([lo, hi], [yi, yi], color=COLORS[name], lw=2, solid_capstyle="round")
        ax.plot(row["mean"], yi, "o", color=COLORS[name], ms=11, mec=SURFACE, mew=2)
        ax.annotate(
            label(row["mean"], row["ci"]),
            (hi, yi),
            xytext=(9, 0),
            textcoords="offset points",
            va="center",
            fontsize=9.5,
            color=INK_2,
        )
    if reference is not None:
        ax.axvline(reference, color=AXIS, lw=1.2)
    lo, hi = (table["mean"] - table["ci"]).min(), (table["mean"] + table["ci"]).max()
    pad = max(hi - lo, 1e-9)
    ax.set_xlim(lo - 0.15 * pad, hi + 0.6 * pad)
    ax.set_ylim(-0.6, len(table) - 0.4)
    ax.set_yticks(y, table.index)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 5, 10]))
    ax.tick_params(axis="y", length=0)
    ax.grid(axis="x", color=GRID, lw=1)
    ax.set_xlabel(xlabel)
    ax.set_title(title)

## Experiment 1: solo

A strategy alone goes through the deck: 51 guesses per game, so the score is also its hit rate
times 51. A small recorder wraps each strategy to tally, per card on the table, how often its
guess was right (it relies on the solo setting: the card after a guess is always the next one
the strategy sees; the last guess of each game is dropped).

In [ ]:
class Recorder:
    """Wraps a strategy and tallies, per table-card rank, how often its guesses are right."""

    def __init__(self, strategy):
        self.strategy = strategy
        self.tries = np.zeros(14)
        self.hits = np.zeros(14)
        self._last = None  # (table rank, guess) waiting for the next card

    def act(self, observation, rng):
        if self._last is not None and len(observation.seen) > 1:  # a new game starts with 1 card
            rank, guess = self._last
            drawn = observation.table_card.rank
            self.tries[rank] += 1
            self.hits[rank] += drawn == rank or (drawn > rank) == (guess is Guess.HIGHER)
        guess = self.strategy.act(observation, rng)
        self._last = (observation.table_card.rank, guess)
        return guess


recorders = {name: Recorder(strategy) for name, strategy in STRATEGIES.items()}
solo = pd.DataFrame(
    {name: simulate(game, [rec], N_SOLO, SEED)[:, 0] for name, rec in recorders.items()}
)
hit_rate = pd.DataFrame(
    {name: rec.hits[1:] / rec.tries[1:] for name, rec in recorders.items()}, index=range(1, 14)
)

solo_score = summary(solo)
solo_vs_trivial = gap_to(solo, "trivial", omit=["random"])
solo_vs_optimal = gap_to(solo, "optimal counting", omit=["random"])
display(
    solo_score.assign(hit_rate=solo_score["mean"] / 51)
    .rename(columns={"mean": "score", "ci": "± CI"})
    .round(3)
)

In [ ]:
fig = plt.figure(figsize=(15, 9), layout="constrained")
grid = fig.add_gridspec(2, 3, height_ratios=[1, 1.15])
fig.suptitle(
    f"Solo play: {N_SOLO:,} paired games per strategy, {CONFIDENCE:.0%} confidence intervals",
    x=0.01,
    ha="left",
    fontsize=14,
    fontweight="bold",
)
strategy_legend(fig)

dot_plot(
    fig.add_subplot(grid[0, 0]),
    solo_score,
    "Mean score (out of 51 guesses)",
    "Points per game",
    plus_minus,
)
dot_plot(
    fig.add_subplot(grid[0, 1]),
    solo_vs_trivial,
    "Score minus trivial's, same deals",
    "Points per game (random omitted: far below)",
    signed,
    reference=0,
)
dot_plot(
    fig.add_subplot(grid[0, 2]),
    solo_vs_optimal,
    "Gap to optimal counting, same deals",
    "Points per game (random omitted: far below)",
    signed,
    reference=0,
)

ax = fig.add_subplot(grid[1, :])
for name in NAMES:
    ax.plot(
        hit_rate.index,
        hit_rate[name],
        color=COLORS[name],
        lw=2,
        marker="o",
        ms=7,
        mec=SURFACE,
        mew=1.5,
    )
ranks = ["A", *range(2, 11), "J", "Q", "K"]
ax.set_xticks(hit_rate.index, ranks)
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.grid(axis="y", color=GRID, lw=1)
ax.set_xlabel("Card on the table")
ax.set_ylabel("Share of guesses that were right")
ax.set_title("Hit rate by card on the table (lines overlap where strategies guess alike)")
plt.show()

## Experiment 2: multi-player

Five players, one per strategy. Every distinct circular seating plays the same deals (the starting
player is random, so rotations are equivalent); per-strategy scores and win shares are then
averaged over the seatings, game by game.

In [ ]:
orders = [(NAMES[0], *rest) for rest in permutations(NAMES[1:])]
scores_by_order, wins_by_order = {}, {}
for order in orders:
    scores = simulate(game, [STRATEGIES[name] for name in order], N_MULTI, SEED)
    scores_by_order[order] = pd.DataFrame(scores, columns=order)[NAMES]
    wins_by_order[order] = pd.DataFrame(win_shares(scores), columns=order)[NAMES]

multi_score = sum(scores_by_order.values()) / len(orders)  # game by game, averaged over seatings
multi_win = sum(wins_by_order.values()) / len(orders)

# Mean score of each strategy depending on who sits right before it, as a deviation from its own
# average: the player after a miss inherits the card that beat the previous one.
by_predecessor = pd.DataFrame(np.nan, index=NAMES, columns=NAMES)
for name in NAMES:
    for before in NAMES:
        if before != name:
            by_predecessor.loc[name, before] = np.mean(
                [scores_by_order[o][name].mean() for o in orders if o[o.index(name) - 1] == before]
            )
seat_effect = by_predecessor.sub(multi_score.mean(), axis=0)

score_table, win_table = summary(multi_score), summary(multi_win)
multi_vs_trivial = gap_to(multi_score, "trivial", omit=["random"])
multi_vs_optimal = gap_to(multi_score, "optimal counting", omit=["random"])
display(
    pd.concat({"score": score_table, "win rate": win_table}, axis=1)
    .rename(columns={"ci": "± CI"})
    .round(3)
)

In [ ]:
def seat_heatmap(fig, ax, effect):
    """Deviation of each strategy's score (row) by the strategy seated right before it (column)."""
    cmap = LinearSegmentedColormap.from_list("diverging", ["#e34948", "#f0efec", "#2a78d6"])
    cmap.set_bad(SURFACE)
    values = effect.to_numpy()
    limit = np.nanmax(np.abs(values))
    image = ax.imshow(values, cmap=cmap, vmin=-limit, vmax=limit, aspect="auto")
    for i in range(len(effect)):
        for j in range(len(effect)):
            v = values[i, j]
            if np.isnan(v):
                ax.text(j, i, "-", ha="center", va="center", color=AXIS, fontsize=11)
            else:
                ax.text(
                    j,
                    i,
                    f"{v:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=10.5,
                    color="white" if abs(v) > 0.6 * limit else INK,
                )
    ax.set_xticks(range(len(effect)), effect.columns, rotation=20, ha="right")
    ax.set_yticks(range(len(effect)), effect.index)
    ax.set_xticks(np.arange(-0.5, len(effect)), minor=True)
    ax.set_yticks(np.arange(-0.5, len(effect)), minor=True)
    ax.grid(which="minor", color=SURFACE, lw=2)
    ax.tick_params(which="both", length=0)
    ax.spines[:].set_visible(False)
    ax.set_xlabel("Strategy seated right before it (columns)")
    ax.set_title("Seat order: who sits right before it")
    bar = fig.colorbar(image, ax=ax, shrink=0.75, pad=0.03)
    bar.set_label("Points vs. own average (blue: better)", fontsize=9)
    bar.outline.set_visible(False)


equal_share = 1 / len(NAMES)
fig = plt.figure(figsize=(15, 9.5), layout="constrained")
grid = fig.add_gridspec(2, 3)
fig.suptitle(
    f"{len(NAMES)}-player games: {N_MULTI:,} paired games per seating x {len(orders)} circular "
    f"seatings, {CONFIDENCE:.0%} confidence intervals",
    x=0.01,
    ha="left",
    fontsize=14,
    fontweight="bold",
)
strategy_legend(fig)

dot_plot(fig.add_subplot(grid[0, 0]), score_table, "Mean score", "Points per game", plus_minus)
ax = fig.add_subplot(grid[0, 1])
dot_plot(
    ax,
    win_table,
    f"Win rate (ties split; equal share is {equal_share:.0%})",
    "Share of games won",
    percent,
    reference=equal_share,
)
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
dot_plot(
    fig.add_subplot(grid[0, 2]),
    multi_vs_optimal,
    "Gap to optimal counting, same deals",
    "Points per game (random omitted: far below)",
    signed,
    reference=0,
)
dot_plot(
    fig.add_subplot(grid[1, 0]),
    multi_vs_trivial,
    "Score minus trivial's, same deals",
    "Points per game (random omitted: far below)",
    signed,
    reference=0,
)
seat_heatmap(fig, fig.add_subplot(grid[1, 1:]), seat_effect)
plt.show()